In [135]:
pip install pandas matplotlib jupyter nbformat

Note: you may need to restart the kernel to use updated packages.


In [136]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [137]:
brent_crude = pd.DataFrame({
    'year': [2021, 2022, 2023, 2024, 2025, 2026],
    'brent_avg_usd_bbl': [71, 100, 82, 82, 68, 80]
})
brent_crude

,year,brent_avg_usd_bbl
0,2021,71
1,2022,100
2,2023,82
3,2024,82
4,2025,68
5,2026,80


In [138]:
cedi_usd = pd.DataFrame({
    'year': [2021, 2022, 2023, 2024, 2025, 2026],
    'ghs_per_usd_avg': [5.8, 8.6, 11.9, 14.7, 12.7, 11.4]
})
cedi_usd

,year,ghs_per_usd_avg
0,2021,5.8
1,2022,8.6
2,2023,11.9
3,2024,14.7
4,2025,12.7
5,2026,11.4


In [139]:
local_pump_price = pd.DataFrame({
    'year': [2021, 2022, 2023, 2024, 2025, 2026],
    'petrol_floor_ghs_l': [6.5, 11.5, 13.5, 15.0, 13.5, 14.53],
    'diesel_floor_ghs_l': [6.2, 12.0, 13.8, 15.8, 13.2, 14.97]
})
local_pump_price

,year,petrol_floor_ghs_l,diesel_floor_ghs_l
0,2021,6.50,6.20
1,2022,11.50,12.00
2,2023,13.50,13.80
3,2024,15.00,15.80
4,2025,13.50,13.20
5,2026,14.53,14.97


In [140]:
subsidy_events = pd.DataFrame({
    'year': [2015, 2026, 2026],
    'event': ['Blanket fuel subsidies ended', 'Regulatory margin cut on diesel and petrol (April)', 'One-month diesel margin cut, Cabinet-approved (August 4)'],
    'product_affected': ['Petrol and Diesel', 'Diesel and Petrol', 'Diesel']
})
subsidy_events

,year,event,product_affected
0,2015,Blanket fuel subsidies ended,Petrol and Diesel
1,2026,Regulatory margin cut on diesel and petrol (Ap...,Diesel and Petrol
2,2026,"One-month diesel margin cut, Cabinet-approved ...",Diesel


In [141]:
combined_trend = brent_crude.merge(cedi_usd, on='year').merge(local_pump_price, on='year')
combined_trend

,year,brent_avg_usd_bbl,ghs_per_usd_avg,petrol_floor_ghs_l,diesel_floor_ghs_l
0,2021,71,5.8,6.50,6.20
1,2022,100,8.6,11.50,12.00
2,2023,82,11.9,13.50,13.80
3,2024,82,14.7,15.00,15.80
4,2025,68,12.7,13.50,13.20
5,2026,80,11.4,14.53,14.97


In [142]:
combined_trend['brent_yoy_pct'] = combined_trend['brent_avg_usd_bbl'].pct_change() * 100
combined_trend['cedi_yoy_pct'] = combined_trend['ghs_per_usd_avg'].pct_change() * 100
combined_trend['petrol_yoy_pct'] = combined_trend['petrol_floor_ghs_l'].pct_change() * 100
combined_trend['diesel_yoy_pct'] = combined_trend['diesel_floor_ghs_l'].pct_change() * 100
combined_trend.round(2)

,year,brent_avg_usd_bbl,ghs_per_usd_avg,petrol_floor_ghs_l,diesel_floor_ghs_l,brent_yoy_pct,cedi_yoy_pct,petrol_yoy_pct,diesel_yoy_pct
0,2021,71,5.8,6.50,6.20,NaN,NaN,NaN,NaN
1,2022,100,8.6,11.50,12.00,40.85,48.28,76.92,93.55
2,2023,82,11.9,13.50,13.80,-18.00,38.37,17.39,15.00
3,2024,82,14.7,15.00,15.80,0.00,23.53,11.11,14.49
4,2025,68,12.7,13.50,13.20,-17.07,-13.61,-10.00,-16.46
5,2026,80,11.4,14.53,14.97,17.65,-10.24,7.63,13.41


In [143]:
correlation_matrix = combined_trend[['brent_yoy_pct', 'cedi_yoy_pct', 'petrol_yoy_pct', 'diesel_yoy_pct']].corr()
correlation_matrix.round(2)

,brent_yoy_pct,cedi_yoy_pct,petrol_yoy_pct,diesel_yoy_pct
brent_yoy_pct,1.00,0.33,0.80,0.86
cedi_yoy_pct,0.33,1.00,0.79,0.75
petrol_yoy_pct,0.80,0.79,1.00,1.00
diesel_yoy_pct,0.86,0.75,1.00,1.00


In [144]:
valid_years = combined_trend.dropna()
petrol_brent_elasticity = (valid_years['petrol_yoy_pct'] / valid_years['brent_yoy_pct']).mean()
petrol_cedi_elasticity = (valid_years['petrol_yoy_pct'] / valid_years['cedi_yoy_pct']).mean()
diesel_brent_elasticity = (valid_years['diesel_yoy_pct'] / valid_years['brent_yoy_pct']).mean()
diesel_cedi_elasticity = (valid_years['diesel_yoy_pct'] / valid_years['cedi_yoy_pct']).mean()

print(f"Petrol elasticity to Brent: {petrol_brent_elasticity:.2f}")
print(f"Petrol elasticity to Cedi: {petrol_cedi_elasticity:.2f}")
print(f"Diesel elasticity to Brent: {diesel_brent_elasticity:.2f}")
print(f"Diesel elasticity to Cedi: {diesel_cedi_elasticity:.2f}")

Petrol elasticity to Brent: inf
Petrol elasticity to Cedi: 0.50
Diesel elasticity to Brent: inf
Diesel elasticity to Cedi: 0.57


In [145]:
import numpy as np

def elasticity(x, y):
    slope, intercept = np.polyfit(x, y, 1)
    return slope

petrol_brent_elasticity = elasticity(valid_years['brent_yoy_pct'], valid_years['petrol_yoy_pct'])
petrol_cedi_elasticity = elasticity(valid_years['cedi_yoy_pct'], valid_years['petrol_yoy_pct'])
diesel_brent_elasticity = elasticity(valid_years['brent_yoy_pct'], valid_years['diesel_yoy_pct'])
diesel_cedi_elasticity = elasticity(valid_years['cedi_yoy_pct'], valid_years['diesel_yoy_pct'])

print(f"Petrol elasticity to Brent: {petrol_brent_elasticity:.2f}")
print(f"Petrol elasticity to Cedi: {petrol_cedi_elasticity:.2f}")
print(f"Diesel elasticity to Brent: {diesel_brent_elasticity:.2f}")
print(f"Diesel elasticity to Cedi: {diesel_cedi_elasticity:.2f}")

Petrol elasticity to Brent: 1.07
Petrol elasticity to Cedi: 0.93
Diesel elasticity to Brent: 1.41
Diesel elasticity to Cedi: 1.09


In [146]:
current_momentum = pd.DataFrame({
    'metric': ['Brent, 1-month change (as of Aug 5, 2026)', 'Cedi, 6-month change (as of Aug 2026)'],
    'pct_change': [11.64, -7.26],
    'current_level': [80.37, 11.66]
})
current_momentum

,metric,pct_change,current_level
0,"Brent, 1-month change (as of Aug 5, 2026)",11.64,80.37
1,"Cedi, 6-month change (as of Aug 2026)",-7.26,11.66


In [147]:
# Convert cedi momentum to cedis-per-dollar direction (consistent with our historical data)
cedi_usd_value_change = -7.26  # cedi weakened 7.26% in USD-per-cedi terms
cedi_per_dollar_change = cedi_usd_value_change / (1 + cedi_usd_value_change/100) * -1
print(f"Cedi weakening, cedis-per-dollar terms: {cedi_per_dollar_change:.2f}%")

# Spread current momentum across the next 3 bi-weekly pricing windows
brent_window_pct = 11.64 / 3
cedi_window_pct = cedi_per_dollar_change / 13  # 6 months ≈ 13 bi-weekly windows

# Apply elasticities per window
petrol_window_change = petrol_brent_elasticity * brent_window_pct + petrol_cedi_elasticity * cedi_window_pct
diesel_window_change = diesel_brent_elasticity * brent_window_pct + diesel_cedi_elasticity * cedi_window_pct

print(f"Petrol projected change per window: {petrol_window_change:.2f}%")
print(f"Diesel projected change per window: {diesel_window_change:.2f}%")

# Build the 3-window forecast table
current_petrol = 14.53
current_diesel = 14.97

forecast = pd.DataFrame({
    'window': ['Current (Aug 4-15, 2026)', 'Window +1', 'Window +2', 'Window +3'],
    'petrol_ghs_l': [current_petrol,
                      current_petrol * (1 + petrol_window_change/100),
                      current_petrol * (1 + petrol_window_change/100)**2,
                      current_petrol * (1 + petrol_window_change/100)**3],
    'diesel_ghs_l': [current_diesel,
                      current_diesel * (1 + diesel_window_change/100),
                      current_diesel * (1 + diesel_window_change/100)**2,
                      current_diesel * (1 + diesel_window_change/100)**3]
})
forecast.round(2)

Cedi weakening, cedis-per-dollar terms: 7.83%
Petrol projected change per window: 4.70%
Diesel projected change per window: 6.13%


,window,petrol_ghs_l,diesel_ghs_l
0,"Current (Aug 4-15, 2026)",14.53,14.97
1,Window +1,15.21,15.89
2,Window +2,15.93,16.86
3,Window +3,16.67,17.90


In [148]:
forecast['diesel_with_intervention_ghs_l'] = forecast['diesel_ghs_l'] * 0.97  # approximate 3% margin relief, consistent with the August 2026 cut
forecast.round(2)

,window,petrol_ghs_l,diesel_ghs_l,diesel_with_intervention_ghs_l
0,"Current (Aug 4-15, 2026)",14.53,14.97,14.52
1,Window +1,15.21,15.89,15.41
2,Window +2,15.93,16.86,16.36
3,Window +3,16.67,17.90,17.36


In [149]:
taxes_and_levies = pd.DataFrame({
    'levy': ['Energy Debt Recovery Levy', 'Road Fund Levy', 'Special Petroleum Tax', 'Energy Sector Levy', 'Other levies (aggregate)'],
    'rate_ghs_per_litre': [0.49, 0.48, 0.46, 1.00, 0.47],
    'flows_to': ['Consolidated Fund', 'Consolidated Fund', 'Consolidated Fund', 'Consolidated Fund', 'Consolidated Fund']
})
taxes_and_levies

,levy,rate_ghs_per_litre,flows_to
0,Energy Debt Recovery Levy,0.49,Consolidated Fund
1,Road Fund Levy,0.48,Consolidated Fund
2,Special Petroleum Tax,0.46,Consolidated Fund
3,Energy Sector Levy,1.00,Consolidated Fund
4,Other levies (aggregate),0.47,Consolidated Fund


In [150]:
omc_uppf_margins = pd.DataFrame({
    'margin': ['BOST Margin', 'UPPF, Primary Distribution, Fuel Marking, and OMC margins (aggregate)'],
    'rate_ghs_per_litre': [0.12, 1.25],
    'flows_to': ['BOST', 'UPPF, BOST, NPA, OMCs']
})
omc_uppf_margins

,margin,rate_ghs_per_litre,flows_to
0,BOST Margin,0.12,BOST
1,"UPPF, Primary Distribution, Fuel Marking, and ...",1.25,"UPPF, BOST, NPA, OMCs"


In [151]:
price_floors = pd.DataFrame({
    'window': ['1-15 Apr 2026', '16-30 Apr 2026', '1-15 May 2026', '2nd window Jul 2026', '1st window Aug 2026', '2nd window Aug 2026 (4-15 Aug)'],
    'petrol_floor_ghs_l': [None, None, 13.25, 13.28, 14.53, 14.53],
    'diesel_floor_ghs_l': [None, None, 14.30, 14.35, 16.97, 14.97],
    'note': ['Peak of 6-window rise, pre-relief', 'First window after Cabinet-directed margin/tax relief', 'Relief in effect', 'Normal window', 'Normal window', 'One-month diesel margin cut in effect']
})
price_floors

,window,petrol_floor_ghs_l,diesel_floor_ghs_l,note
0,1-15 Apr 2026,NaN,NaN,"Peak of 6-window rise, pre-relief"
1,16-30 Apr 2026,NaN,NaN,First window after Cabinet-directed margin/tax...
2,1-15 May 2026,13.25,14.30,Relief in effect
3,2nd window Jul 2026,13.28,14.35,Normal window
4,1st window Aug 2026,14.53,16.97,Normal window
5,2nd window Aug 2026 (4-15 Aug),14.53,14.97,One-month diesel margin cut in effect


In [152]:
levy_margin_share = pd.DataFrame({
    'window': ['1-15 Apr 2026', '1-15 May 2026'],
    'petrol_share_pct': [31.04, 28.79],
    'diesel_share_pct': [24.26, 13.93],
    'lpg_share_pct': [12.40, 13.23]
})
levy_margin_share

,window,petrol_share_pct,diesel_share_pct,lpg_share_pct
0,1-15 Apr 2026,31.04,24.26,12.40
1,1-15 May 2026,28.79,13.93,13.23


In [153]:
import os
os.makedirs('ghana-fuel-price-dataset', exist_ok=True)

brent_crude.to_csv('ghana-fuel-price-dataset/brent_crude.csv', index=False)
cedi_usd.to_csv('ghana-fuel-price-dataset/cedi_usd.csv', index=False)
local_pump_price.to_csv('ghana-fuel-price-dataset/local_pump_price.csv', index=False)
subsidy_events.to_csv('ghana-fuel-price-dataset/subsidy_events.csv', index=False)
combined_trend.to_csv('ghana-fuel-price-dataset/combined_trend.csv', index=False)
correlation_matrix.to_csv('ghana-fuel-price-dataset/correlation_matrix.csv')
current_momentum.to_csv('ghana-fuel-price-dataset/current_momentum.csv', index=False)
forecast.to_csv('ghana-fuel-price-dataset/forecast_v2.csv', index=False)
taxes_and_levies.to_csv('ghana-fuel-price-dataset/taxes_and_levies.csv', index=False)
omc_uppf_margins.to_csv('ghana-fuel-price-dataset/omc_uppf_margins.csv', index=False)
price_floors.to_csv('ghana-fuel-price-dataset/price_floors.csv', index=False)
levy_margin_share.to_csv('ghana-fuel-price-dataset/levy_margin_share.csv', index=False)

print("All 12 files saved.")
print(sorted(os.listdir('ghana-fuel-price-dataset')))

All 12 files saved.
['brent_crude.csv', 'cedi_usd.csv', 'combined_trend.csv', 'correlation_matrix.csv', 'current_momentum.csv', 'data_sources.csv', 'forecast.csv', 'forecast_v2.csv', 'levy_margin_share.csv', 'local_pump_price.csv', 'omc_uppf_margins.csv', 'price_buildup_funnel.csv', 'price_composition.csv', 'price_composition_full.csv', 'price_floors.csv', 'product_consumption_share.csv', 'subsidy_events.csv', 'taxes_and_levies.csv']


In [154]:
forecast['petrol_wow_pct'] = forecast['petrol_ghs_l'].pct_change() * 100
forecast['diesel_wow_pct'] = forecast['diesel_ghs_l'].pct_change() * 100
forecast['diesel_savings_ghs'] = forecast['diesel_ghs_l'] - forecast['diesel_with_intervention_ghs_l']
forecast.round(2)

,window,petrol_ghs_l,diesel_ghs_l,diesel_with_intervention_ghs_l,petrol_wow_pct,diesel_wow_pct,diesel_savings_ghs
0,"Current (Aug 4-15, 2026)",14.53,14.97,14.52,NaN,NaN,0.45
1,Window +1,15.21,15.89,15.41,4.7,6.13,0.48
2,Window +2,15.93,16.86,16.36,4.7,6.13,0.51
3,Window +3,16.67,17.90,17.36,4.7,6.13,0.54


In [155]:
price_composition = pd.DataFrame({
    'product': ['Petrol', 'Diesel'],
    'base_cost_ghs': [14.53 - 2.90 - 1.37, 14.97 - 2.90 - 1.37],
    'taxes_ghs': [2.90, 2.90],
    'margins_ghs': [1.37, 1.37]
})
price_composition

,product,base_cost_ghs,taxes_ghs,margins_ghs
0,Petrol,10.26,2.9,1.37
1,Diesel,10.70,2.9,1.37


In [156]:
forecast.to_csv('ghana-fuel-price-dataset/forecast_v2.csv', index=False)
price_composition.to_csv('ghana-fuel-price-dataset/price_composition.csv', index=False)
print("Saved updated forecast.csv")
print("Saved price_composition.csv")

Saved updated forecast.csv
Saved price_composition.csv


In [157]:
# LPG price composition (derived from the 13.23% May 2026 tax+margin share, split in the same ratio as petrol/diesel)
lpg_total = 11.06
lpg_tax_margin = lpg_total * 0.1323
lpg_taxes = lpg_tax_margin * (2.90 / (2.90 + 1.37))
lpg_margins = lpg_tax_margin * (1.37 / (2.90 + 1.37))
lpg_base = lpg_total - lpg_tax_margin

price_composition_full = pd.concat([
    price_composition,
    pd.DataFrame({'product': ['LPG'], 'base_cost_ghs': [round(lpg_base,2)], 'taxes_ghs': [round(lpg_taxes,2)], 'margins_ghs': [round(lpg_margins,2)]})
], ignore_index=True)
price_composition_full

,product,base_cost_ghs,taxes_ghs,margins_ghs
0,Petrol,10.26,2.90,1.37
1,Diesel,10.70,2.90,1.37
2,LPG,9.60,0.99,0.47


In [158]:
product_consumption_share = pd.DataFrame({
    'product': ['Petrol', 'Diesel', 'LPG'],
    'share_of_total_consumption_pct': [49.35, 36.11, 5.27],
    'volume_2025': [3.10, 2.76, 0.376],
    'unit': ['billion litres', 'billion litres', 'billion kg']
})
product_consumption_share

,product,share_of_total_consumption_pct,volume_2025,unit
0,Petrol,49.35,3.100,billion litres
1,Diesel,36.11,2.760,billion litres
2,LPG,5.27,0.376,billion kg


In [159]:
price_buildup_funnel = pd.DataFrame({
    'stage': ['Base Cost (World Price + Import)', 'Base + Margins', 'Final Pump Price (+ Taxes)'],
    'petrol_ghs': [10.26, 10.26 + 1.37, 14.53],
    'diesel_ghs': [10.70, 10.70 + 1.37, 14.97]
})
price_buildup_funnel

,stage,petrol_ghs,diesel_ghs
0,Base Cost (World Price + Import),10.26,10.70
1,Base + Margins,11.63,12.07
2,Final Pump Price (+ Taxes),14.53,14.97


In [160]:
price_composition_full.to_csv('ghana-fuel-price-dataset/price_composition_full.csv', index=False)
product_consumption_share.to_csv('ghana-fuel-price-dataset/product_consumption_share.csv', index=False)
price_buildup_funnel.to_csv('ghana-fuel-price-dataset/price_buildup_funnel.csv', index=False)
print("Saved all three")

Saved all three


In [161]:
check = pd.read_csv('ghana-fuel-price-dataset/forecast.csv')
check.columns.tolist()

['window',
 'petrol_ghs_l',
 'diesel_ghs_l',
 'diesel_with_intervention_ghs_l',
 'petrol_wow_pct',
 'diesel_wow_pct',
 'diesel_savings_ghs']

In [162]:
forecast.to_csv('ghana-fuel-price-dataset/forecast_v2.csv', index=False)
print("Saved as forecast_v2.csv")

Saved as forecast_v2.csv


In [163]:
forecast.to_csv('ghana-fuel-price-dataset/forecast_v2.csv', index=False)
print("Saved as forecast_v2.csv")

Saved as forecast_v2.csv


In [164]:
data_sources = pd.DataFrame({
    'dataset': [
        'Brent crude, 5-year annual average', 'Cedi/USD, 5-year annual average',
        'Local pump price benchmarks', 'Combined trend and correlation',
        'Current momentum and forecast', 'Taxes and levies',
        'OMC and UPPF margins', 'Price floors by pricing window',
        'Levy and margin share by window', 'Subsidy and intervention events',
        'Price composition by product (incl. LPG)', 'Product consumption share'
    ],
    'source': [
        'EIA / World of Statistics, compiled annual averages', 'exchange-rates.org, TheGlobalEconomy.com, compiled annual averages',
        'NPA pricing window announcements, press coverage', 'Derived from Brent, cedi, and pump price tables above',
        'TradingEconomics (Brent), press coverage (cedi 6-month trend)', 'NPA price build-up structure, CBOD market outlook reports',
        'NPA price build-up structure, CBOD market outlook reports', 'NPA pricing window announcements, April-August 2026',
        'CBOD Ghana market outlook reports, April and May 2026 windows', 'NPA policy history; Cabinet-approved 2026 relief measures',
        'Derived from taxes_and_levies and omc_uppf_margins; LPG share estimated from levy_margin_share', 'Chamber of Oil Marketing Companies (COMAC) 2025 downstream sector report'
    ],
    'date_pulled': ['2026-08-06'] * 12
})
data_sources.to_csv('ghana-fuel-price-dataset/data_sources.csv', index=False)
print("Saved data_sources.csv")

Saved data_sources.csv


# Ghana Fuel Price Analysis: 5-Year Trend, Correlation, and 3-Window Forecast

**Prepared by Richard Courage Cobbinah**

Sources: National Petroleum Authority (NPA) price floors and pricing windows, Chamber for Bulk Oil Distributors (CBOD) Ghana market outlook reports, EIA/World of Statistics Brent crude annual averages, exchange-rates.org and TheGlobalEconomy.com cedi/USD data, press coverage of 2026 fuel relief interventions.

## What this covers

Twelve tables tracing the relationship between world crude prices, the cedi's exchange rate, and Ghana's local pump price, 2021 through 2026, with a build-up breakdown of taxes, levies, and margins, and a momentum-based forecast for the next three pricing windows.

## Files

- `brent_crude.csv`: Brent crude annual average price, 2021-2026.
- `cedi_usd.csv`: Cedi/USD annual average exchange rate, 2021-2026.
- `local_pump_price.csv`: Annual benchmark petrol and diesel floor prices, 2021-2026.
- `combined_trend.csv`: Merged Brent, cedi, and pump price series with year-over-year percentage change for each.
- `correlation_matrix.csv`: Correlation coefficients between Brent, cedi, petrol, and diesel year-over-year movements.
- `current_momentum.csv`: Most recent short-term Brent and cedi momentum (1-month and 6-month, as of August 2026), used as the forecast's starting point.
- `forecast.csv`: Projected petrol and diesel prices for the next 3 pricing windows, plus a scenario column showing diesel price under a repeated margin-relief intervention.
- `taxes_and_levies.csv`: Named fuel taxes and levies, GH¢ per litre, flowing to the Consolidated Fund.
- `omc_uppf_margins.csv`: BOST Margin plus the combined UPPF/OMC/distribution margin.
- `price_floors.csv`: NPA price floors by specific 2026 pricing window, with notes on relief interventions in effect.
- `levy_margin_share.csv`: Taxes, levies, and margins as a percentage of pump price, by window and product, showing the measured effect of the April 2026 relief intervention.
- `subsidy_events.csv`: Confirmed, dated fuel pricing intervention events, 2015-2026.

## Key findings

- **2022 was the sharpest pass-through year on record here**: petrol rose 76.9% and diesel 93.6%, far outpacing the 48.3% cedi depreciation and 40.9% Brent increase that year, a compounding effect where currency and crude moved against Ghana simultaneously.
- **Diesel is structurally more sensitive than petrol** to both Brent (elasticity 1.41 vs. 1.07) and the cedi (1.09 vs. 0.93), based on a 4-year regression fit.
- **The April 2026 relief intervention measurably worked**: diesel's combined tax, levy, and margin share of pump price fell from 24.26% to 13.93% in a single pricing window, the clearest evidence in this dataset that targeted margin suspension, not tax cuts, was the primary relief lever used.
- **Forecast**: based on current Brent momentum (+11.64% over the past month) and cedi momentum (weakening over the past 6 months), petrol is projected to rise roughly 4.7% per pricing window and diesel roughly 6.1%, reaching approximately GH¢16.67/L and GH¢17.90/L respectively by the third window out. A repeat of August's diesel margin cut would hold diesel closer to GH¢17.36/L in that same window.

## Data quality notes

- Annual figures for Brent, cedi, and local pump price are representative averages, not full daily or bi-weekly series. A true 24-window-per-year reconstruction was not feasible from public search access; NPA's actual bi-weekly price tables are served as JavaScript-loaded PDFs, not indexed for search or direct retrieval.
- The correlation and elasticity figures are built on only 5 year-over-year data points. Treat the direction (diesel more volatile than petrol, both tracking Brent and the cedi meaningfully) as reliable; treat the exact coefficients as indicative, not statistically robust.
- **BIDEC premiums were not included as a standalone table.** Bulk Import, Distribution and Export Companies price off an "Ex-ref price indicator" referencing international benchmarks plus a CBOD breakeven premium, but no public source discloses this premium as an isolated GH¢/litre figure. It is embedded in, not separate from, the published price build-up.
- The forecast is a momentum-and-elasticity projection, not an econometric model. It is directional, useful for understanding likely trend and scale of movement, not a precise prediction of exact pricing-window figures.
- `taxes_and_levies.csv` and `omc_uppf_margins.csv` include an "aggregate" line for components not individually disclosed in public reporting (several smaller named levies within the GH¢2.90 tax total, and UPPF/distribution margins within the GH¢1.37 margin total).

This report is for informational purposes only and does not constitute financial or trading advice.